# 05. Context rot 측정

입력 길이와 근거 위치를 바꾸되 질문, 모델, 지시, 평가 기준은 고정해야 길이 효과를 비교할 수 있습니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none", max_tokens=300)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 같은 근거를 짧은 입력과 긴 입력에 배치합니다

긴 입력에는 질문과 무관한 문장을 추가합니다. 실제 실험에서는 길이와 위치별로 여러 번 반복합니다.

In [3]:
evidence = "[E01 p.2] 정기 휴관일은 매주 월요일입니다."
question = "미르시립도서관의 정기 휴관일과 출처를 알려 주세요."
noise = "회의 준비 항목을 확인했습니다. " * 120
prompts = {
    "short": f"근거:\n{evidence}\n\n질문: {question}",
    "long": f"근거:\n{noise}\n{evidence}\n\n질문: {question}",
}
print({name: len(text) for name, text in prompts.items()})

{'short': 66, 'long': 2227}


## 3. 두 조건의 실제 사용량을 측정합니다

두 호출의 입력 토큰과 답을 나란히 출력합니다. 두 번의 호출만으로 일반 결론을 내리지는 않습니다.

In [4]:
results = {}
for name, prompt in prompts.items():
    messages = [("system", "근거만 사용하고 출처를 표시하세요."), ("user", prompt)]
    result = model.invoke(messages)
    usage = result.usage_metadata
    if usage is None:
        raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
    results[name] = result
    print(name, "input_tokens:", usage["input_tokens"])
    print(result.text)

short input_tokens: 65
미르시립도서관의 정기 휴관일은 **매주 월요일**입니다.  
출처: **[E01 p.2]**


long input_tokens: 1145
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
출처: **[E01 p.2]**


## 4. 최소 루브릭으로 확인합니다

자동 문자열 검사는 빠르지만, 대상 기관과 문장의 모순 여부는 사람이 함께 읽어야 합니다.

In [5]:
for name, result in results.items():
    answer = result.text
    score = int("월요일" in answer) + int("E01 p.2" in answer)
    print(name, "자동 점수:", score, "/ 2")

short 자동 점수: 2 / 2
long 자동 점수: 2 / 2


## 정리

이번 두 호출은 측정 방법을 익히는 최소 예제입니다. 반복 횟수와 실행 순서를 기록해야 재현 가능한 실험이 됩니다.

## 실패 재현: 조건마다 질문을 바꿉니다

입력 길이와 질문 난이도를 함께 바꾸면 어느 변화가 점수 차이를 만들었는지 구분할 수 없습니다.

In [6]:
def require_same_question(questions: list[str]) -> None:
    if len(set(questions)) != 1:
        raise ValueError("조건마다 질문이 다릅니다.")

try:
    require_same_question([question, "대출 기간은 얼마인가요?"])
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("질문 불일치 오류가 발생해야 합니다.")

관찰한 실패: 조건마다 질문이 다릅니다.


## 수정: 모든 조건에 같은 질문을 사용합니다

짧은 입력과 긴 입력 모두 앞에서 정의한 휴관일 질문을 사용합니다.

In [7]:
fixed_questions = [question, question]
require_same_question(fixed_questions)
assert len(set(fixed_questions)) == 1
print("수정 후 질문 수:", len(fixed_questions))

수정 후 질문 수: 2
